In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import os

In [5]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    api_key=GOOGLE_API_KEY
)

In [7]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [8]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [9]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [10]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why did the football coach bring string to the game?\n\nTo tie the score!',
 'explanation': 'This is a classic pun! The humor comes from the word "tie" having two different meanings:\n\n1.  **"Tie" (literal meaning related to string):** When you have string, you can use it to *tie* knots, *tie* two things together, or fasten something. This is the literal action you\'d perform with string.\n\n2.  **"Tie the score" (idiomatic meaning in sports):** In sports, when teams "tie the score," it means they have an equal number of points. For example, if one team has 7 points and the other scores to get 7 points, the score is tied.\n\nThe joke plays on the absurdity of a coach *literally* bringing string to "tie" the score, when what he (and everyone else) actually wants is for his team to make the score equal in the game. The unexpected literal interpretation of the phrase "tie the score" is what makes it funny.'}

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach bring string to the game?\n\nTo tie the score!', 'explanation': 'This is a classic pun! The humor comes from the word "tie" having two different meanings:\n\n1.  **"Tie" (literal meaning related to string):** When you have string, you can use it to *tie* knots, *tie* two things together, or fasten something. This is the literal action you\'d perform with string.\n\n2.  **"Tie the score" (idiomatic meaning in sports):** In sports, when teams "tie the score," it means they have an equal number of points. For example, if one team has 7 points and the other scores to get 7 points, the score is tied.\n\nThe joke plays on the absurdity of a coach *literally* bringing string to "tie" the score, when what he (and everyone else) actually wants is for his team to make the score equal in the game. The unexpected literal interpretation of the phrase "tie the score" is what makes it funny.'}, next=(), config={'configurab

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach bring string to the game?\n\nTo tie the score!', 'explanation': 'This is a classic pun! The humor comes from the word "tie" having two different meanings:\n\n1.  **"Tie" (literal meaning related to string):** When you have string, you can use it to *tie* knots, *tie* two things together, or fasten something. This is the literal action you\'d perform with string.\n\n2.  **"Tie the score" (idiomatic meaning in sports):** In sports, when teams "tie the score," it means they have an equal number of points. For example, if one team has 7 points and the other scores to get 7 points, the score is tied.\n\nThe joke plays on the absurdity of a coach *literally* bringing string to "tie" the score, when what he (and everyone else) actually wants is for his team to make the score equal in the game. The unexpected literal interpretation of the phrase "tie the score" is what makes it funny.'}, next=(), config={'configura

In [14]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket team bring a ladder to the match?\n...Because they heard the pitch was high!',
 'explanation': 'This joke is a classic pun that plays on the different meanings of the words "pitch" and "high."\n\nHere\'s the breakdown:\n\n1.  **"Pitch" (Meaning 1 - Cricket):** In cricket, the "pitch" refers to the rectangular strip of ground in the center of the field where the bowling and batting take place. This pitch is, of course, flat on the ground and not physically elevated.\n\n2.  **"Pitch" (Meaning 2 - Sound):** "Pitch" can also refer to the quality of a sound, specifically how high or low it is (like a high-pitched voice or a low-pitched hum).\n\n3.  **"High" (Meaning 1 - Physical):** A ladder is used to reach something that is physically "high" or elevated.\n\n4.  **"High" (Meaning 2 - Sound):** When referring to sound, "high" means a high frequency or a shrill sound.\n\n**The Joke\'s Punchline:**\n\nThe joke deliberately twists the second m

In [15]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n...Because they heard the pitch was high!', 'explanation': 'This joke is a classic pun that plays on the different meanings of the words "pitch" and "high."\n\nHere\'s the breakdown:\n\n1.  **"Pitch" (Meaning 1 - Cricket):** In cricket, the "pitch" refers to the rectangular strip of ground in the center of the field where the bowling and batting take place. This pitch is, of course, flat on the ground and not physically elevated.\n\n2.  **"Pitch" (Meaning 2 - Sound):** "Pitch" can also refer to the quality of a sound, specifically how high or low it is (like a high-pitched voice or a low-pitched hum).\n\n3.  **"High" (Meaning 1 - Physical):** A ladder is used to reach something that is physically "high" or elevated.\n\n4.  **"High" (Meaning 2 - Sound):** When referring to sound, "high" means a high frequency or a shrill sound.\n\n**The Joke\'s Punchline:**\n\nThe joke deliberately 

In [16]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n...Because they heard the pitch was high!', 'explanation': 'This joke is a classic pun that plays on the different meanings of the words "pitch" and "high."\n\nHere\'s the breakdown:\n\n1.  **"Pitch" (Meaning 1 - Cricket):** In cricket, the "pitch" refers to the rectangular strip of ground in the center of the field where the bowling and batting take place. This pitch is, of course, flat on the ground and not physically elevated.\n\n2.  **"Pitch" (Meaning 2 - Sound):** "Pitch" can also refer to the quality of a sound, specifically how high or low it is (like a high-pitched voice or a low-pitched hum).\n\n3.  **"High" (Meaning 1 - Physical):** A ladder is used to reach something that is physically "high" or elevated.\n\n4.  **"High" (Meaning 2 - Sound):** When referring to sound, "high" means a high frequency or a shrill sound.\n\n**The Joke\'s Punchline:**\n\nThe joke deliberately

***Time Travel***

In [21]:

workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1a9c6c-c4db-64b9-8000-b318d967f3b8"}})

StateSnapshot(values={'topic': 'Cricket'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1a9c6c-c4db-64b9-8000-b318d967f3b8'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-06T07:44:32.145916+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9c6c-c4d8-6453-bfff-7029ecc9254a'}}, tasks=(PregelTask(id='70ac5c99-8e46-452a-2db2-eb759ca63f71', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the cricket team bring a ladder to the match?\n...Because they heard the pitch was high!'}),), interrupts=())

In [22]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f1a9c6c-c4d8-6453-bfff-7029ecc9254a"}})

{'topic': 'Cricket',
 'joke': 'Why did the cricket team bring a ladder to the game?\n\nBecause they heard the *batsman* was going to hit a six!',
 'explanation': 'This joke plays on a **pun** and a **double meaning** of the word "batsman" and the phrase "hit a six" in cricket.\n\nHere\'s the breakdown:\n\n1.  **Cricket Meaning:**\n    *   A **batsman** is the player in cricket who hits the ball.\n    *   To **hit a six** means the batsman has hit the ball so well and so far that it goes over the boundary rope without bouncing. This instantly earns the team 6 runs. Often, these are very high, powerful shots.\n\n2.  **The Pun/Double Meaning:**\n    *   The word "**batsman**" sounds a lot like "**bats, man!**" – referring to the flying mammal bats.\n    *   If you interpret "hit a six" in this *punny* way, it could be imagined as encountering or dealing with *six* actual flying bats.\n\n3.  **Why the Ladder?**\n    *   The joke sets up the scenario with a ladder, which is used to reach hi

In [ ]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the insect cricket get kicked out of the cricket match?\n\nBecause he kept chirping "Howzat!" after every single ball!', 'explanation': 'This joke plays on the **double meaning of the word "cricket"**:\n\n1.  **Cricket (the insect):** This is the small, chirping insect that the joke\'s character is.\n2.  **Cricket (the sport):** This is the bat-and-ball game being played in the "cricket match."\n\nHere\'s the breakdown:\n\n*   **"Why did the insect cricket get kicked out of the cricket match?"**\n    *   The setup immediately uses both meanings, creating the initial confusion and setting up the pun. An insect wouldn\'t normally be playing a sport, so we know something funny is coming.\n\n*   **"Because he kept chirping \'Howzat!\' after every single ball!"**\n    *   **"Chirping":** This is what an *insect* cricket naturally does. It connects the insect to the action.\n    *   **"Howzat!":** This is a key phrase in the *sport*

In [23]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1a9bed-68f7-645a-8001-c46124d4e37c", "checkpoint_ns": ""}}, {'topic':'Tenis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1a9c70-d233-63e7-8000-c2ebc8d7a325'}}

In [24]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tenis'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9c70-d233-63e7-8000-c2ebc8d7a325'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-09-06T07:46:20.919281+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9bed-68f7-645a-8001-c46124d4e37c'}}, tasks=(PregelTask(id='a7e56936-7c9d-dd05-173f-428758207fe7', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the game?\n\nBecause they heard the *batsman* was going to hit a six!', 'explanation': 'This joke plays on a **pun** and a **double meaning** of the word "batsman" and the phrase "hit a six" in cricket.\n\nHere\'s the breakdown:\n\n1.  **Cricket Meaning:**\n    *   A **batsman** is the

**Fault Tolerance**

In [25]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [26]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [27]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [28]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [29]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [30]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [31]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9c73-edbd-6c56-8003-e5a6e16654cc'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-09-06T07:47:44.337812+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9c73-edbb-6ac3-8002-9d19ca20108e'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9c73-edbb-6ac3-8002-9d19ca20108e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-06T07:47:44.336944+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a9c71-6505-6410-8001-5929d717a587'}}, tasks=(PregelTask(id='b7d463a8-9443-23b9-e817-d3da3131288